# P2 — Charger, typer, valeurs impossibles, plancher

**Enchaînement : 1.2 Charger et compter → 1.3 Typer → 1.5 Valeurs impossibles → 1.10 Le plancher**

Phase 1 du projet *Santander Satisfaction* : connaître les données avant de modéliser.
Chaque tâche suit le même plan : **la question**, **la mesure** (le code), **le constat**, **la décision**
que le porteur doit pouvoir défendre seul.

Pour relancer : *Exécuter tout*. Les données sont lues dans `data/` (récupérées par `git pull`), tables entières.

In [1]:
# --- Réglages communs à toute l'équipe ------------------------------------
import time
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_rows", 80)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:.4f}".format)

# Les données sont dans data/ à la racine du dépôt (récupérées par git pull)
DATA = next(p for p in (Path("data"), Path("../data")) if (p / "train.csv").is_file())
CIBLE, ID = "TARGET", "ID"
GRAINE = 42             # même graine pour toute l'équipe : même découpage
PART_VALIDATION = 0.20  # protocole : 80 % apprentissage / 20 % validation, stratifié
PLANCHER = 0.6890       # AUC de la règle métier, donnée par l'énoncé
SENTINELLES = (-999999, 9999999999)
FAMILLES = ("saldo", "imp", "num", "ind", "delta")

t0 = time.perf_counter()
train = pd.read_csv(DATA / "train.csv")
a_decider = pd.read_csv(DATA / "a_decider.csv")
DUREE_CHARGEMENT = time.perf_counter() - t0
VARIABLES = [c for c in train.columns if c not in (ID, CIBLE)]
y = train[CIBLE]


def famille(col):
    if col in (ID, CIBLE):
        return col
    debut = col.split("_")[0]
    return debut if debut in FAMILLES else "isolée"


def pct(x):
    return f"{100 * x:.2f} %"


print(f"train : {train.shape} · a_decider : {a_decider.shape} · lus en {DUREE_CHARGEMENT:.1f} s depuis {DATA}")

train : (60816, 371) · a_decider : (15204, 370) · lus en 0.8 s depuis ../data


## 1.2 — Charger et compter

**Question** : les tables se chargent-elles en entier, combien de lignes, combien de mécontents ? Les chiffres de l'énoncé sont-ils retrouvés ?

In [2]:
print(f"Chargement : {DUREE_CHARGEMENT:.1f} s · mémoire : {train.memory_usage(deep=True).sum() / 1e6:.0f} Mo "
      f"+ {a_decider.memory_usage(deep=True).sum() / 1e6:.0f} Mo")
for nom, df, attendu in (("train", train, (60816, 371)), ("a_decider", a_decider, (15204, 370))):
    print(f"{nom:10s} {df.shape}  attendu {attendu}  {'OK' if df.shape == attendu else 'ÉCART'}")
print(f"Mécontents : {int(y.sum())} / {len(y)} = {pct(y.mean())}")
print(f"Total clients : {len(train) + len(a_decider)} (énoncé : 76 020)")
print(f"ID uniques : train {train[ID].is_unique}, a_decider {a_decider[ID].is_unique}")

Chargement : 0.8 s · mémoire : 181 Mo + 45 Mo
train      (60816, 371)  attendu (60816, 371)  OK
a_decider  (15204, 370)  attendu (15204, 370)  OK
Mécontents : 2406 / 60816 = 3.96 %
Total clients : 76020 (énoncé : 76 020)
ID uniques : train True, a_decider True


**Constat** : tout est chargé en moins d'une seconde (≈ 226 Mo en mémoire). 60 816 + 15 204 = 76 020 clients, 2 406 mécontents (3,96 %) : **exactement les chiffres de l'énoncé**, aucun fichier tronqué.

**Décision** : on charge les tables **entières** (pas d'échantillon) : c'est rapide, et un échantillon fausse les comptages (voir 1.4, les colonnes constantes).

## 1.3 — Typer

**Question** : chaque colonne a-t-elle le bon type ? Y a-t-il du texte, des nombres mal lus, des indicateurs qui ne sont pas 0/1 ?

In [3]:
tout = pd.concat([train.drop(columns=CIBLE), a_decider])
types = pd.DataFrame({"famille": [famille(c) for c in VARIABLES], "type": [str(tout[c].dtype) for c in VARIABLES]})
pd.crosstab(types["famille"], types["type"])

type,float64,int64
famille,,
delta,10,16
imp,39,10
ind,0,75
isolée,1,4
num,0,143
saldo,61,10


In [4]:
flottants_entiers = [c for c in VARIABLES if tout[c].dtype.kind == "f" and (tout[c] % 1 == 0).all()]
print("Colonnes texte                       :", [c for c in VARIABLES if tout[c].dtype == object] or "aucune")
print("Float qui ne contiennent que des entiers :", len(flottants_entiers))
print("ind_ qui ne valent pas 0/1           :",
      [c for c in VARIABLES if famille(c) == "ind" and not set(tout[c].unique()) <= {0, 1}] or "aucune")
reduit = tout.copy()
for c in VARIABLES:
    reduit[c] = pd.to_numeric(reduit[c], downcast="integer" if reduit[c].dtype.kind == "i" else "float")
print(f"Mémoire : {tout.memory_usage(deep=True).sum() / 1e6:.0f} Mo -> {reduit.memory_usage(deep=True).sum() / 1e6:.0f} Mo en réduisant les types")

Colonnes texte                       : aucune
Float qui ne contiennent que des entiers : 0
ind_ qui ne valent pas 0/1           : aucune


Mémoire : 226 Mo -> 94 Mo en réduisant les types


**Constat** : **tout est numérique** (int64 ou float64), aucun texte, aucun nombre mal lu. Les 75 `ind_` valent bien 0 ou 1. Réduire les types ferait passer la mémoire de 226 à 94 Mo.

**Décision** : **aucune conversion nécessaire.** La réduction de mémoire n'est pas utile (le pipeline tient largement en mémoire et en temps) : on garde les types d'origine. Attention : un type numérique ne veut pas dire « quantité » — `var36` est une catégorie codée en nombres (voir P3).

## 1.5 — Valeurs impossibles

**Question** : y a-t-il des valeurs qui ne peuvent pas être de vraies mesures ? Codes sentinelles, comptages négatifs, âges impossibles…

In [5]:
lignes = []
for c in VARIABLES:
    s = tout[c]
    sent = int(s.isin(SENTINELLES).sum())
    neg = int((s < 0).sum()) if famille(c) in ("num", "ind") else 0
    if sent or neg:
        lignes.append((c, famille(c), sent, neg, s.min(), s.max()))
impossibles = pd.DataFrame(lignes, columns=["colonne", "famille", "sentinelles", "négatifs", "min", "max"])
print(f"{len(impossibles)} colonnes concernées, {impossibles['sentinelles'].sum()} cellules sentinelles")
impossibles.sort_values("sentinelles", ascending=False)

27 colonnes concernées, 1310 cellules sentinelles


,colonne,famille,sentinelles,négatifs,min,max
3,delta_imp_aport_var13_1y3,delta,370,0,-1.0000,9999999999.0000
15,delta_num_aport_var13_1y3,delta,370,0,-1.0000,9999999999.0000
0,var3,isolée,116,0,-999999.0000,238.0000
18,delta_num_compra_var44_1y3,delta,70,0,-1.0000,9999999999.0000
6,delta_imp_compra_var44_1y3,delta,70,0,-1.0000,9999999999.0000
26,delta_num_venta_var44_1y3,delta,42,0,-1.0000,9999999999.0000
14,delta_imp_venta_var44_1y3,delta,42,0,-1.0000,9999999999.0000
4,delta_imp_aport_var17_1y3,delta,39,0,-1.0000,9999999999.0000
16,delta_num_aport_var17_1y3,delta,39,0,-1.0000,9999999999.0000
19,delta_num_reemb_var13_1y3,delta,38,0,0.0000,9999999999.0000


In [6]:
print("var3 = -999999 :", int((tout["var3"] == -999999).sum()), "lignes")
print(f"var15 : de {tout['var15'].min()} à {tout['var15'].max()} ; moins de 18 ans : {int((tout['var15'] < 18).sum())} clients "
      f"(dont {int((train['var15'] < 18).sum())} dans train, {int(train.loc[train['var15'] < 18, CIBLE].sum())} mécontent)")
print("Comptages num_ ou indicateurs ind_ négatifs :", int(impossibles["négatifs"].sum()))
neg_saldo = [c for c in VARIABLES if famille(c) == "saldo" and (tout[c] < 0).any()]
print(f"Soldes saldo_ négatifs : {len(neg_saldo)} colonnes (un solde peut être négatif : découvert)")

var3 = -999999 : 116 lignes
var15 : de 5 à 105 ; moins de 18 ans : 212 clients (dont 175 dans train, 0 mécontent)
Comptages num_ ou indicateurs ind_ négatifs : 0
Soldes saldo_ négatifs : 14 colonnes (un solde peut être négatif : découvert)


**Constat**

- Deux codes sentinelles : **`-999999`** dans `var3` (116 clients) et **`9999999999`** dans les 26 colonnes `delta_` (1 194 cellules au total).
- **Aucun** comptage (`num_`) ni indicateur (`ind_`) négatif. Les `-1` des `delta_` ne sont pas des erreurs (variation de −100 %).
- `var15` va de 5 à 105 : c'est un âge ; 212 clients ont moins de 18 ans (comptes de mineurs possibles, **aucun** mécontent parmi eux). Les soldes négatifs (14 colonnes) sont plausibles (découvert).

**Décision**

- On **garde** toutes les lignes : aucune valeur n'est une erreur de saisie à supprimer.
- Les deux sentinelles sont transmises à **P3 (1.6 Trous)**, qui décide de leur traitement : `-999999` est un vrai manquant, `9999999999` n'en est pas un (variation depuis zéro).

## 1.10 — Le plancher

**Question** : quel score obtient le tri que la banque applique aujourd'hui, sans modèle ? L'énoncé donne **AUC 0,6890** : quelle règle simple le reproduit ? C'est le chiffre que tout modèle doit battre.

**Protocole** : découpage **80 / 20 stratifié** sur `TARGET`, graine 42. Pour chaque variable prise seule, le sens est choisi sur l'apprentissage et l'AUC mesurée sur la validation.

In [7]:
# --- Protocole de validation et règle à une variable -------------------------
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

# 80 / 20 stratifié sur TARGET, graine fixe : chaque partie garde 3,96 % de mécontents
i_app, i_val = train_test_split(train.index, test_size=PART_VALIDATION, stratify=y, random_state=GRAINE)

# Chaque variable prise seule comme score. Le sens (grand = mécontent, ou l'inverse)
# est choisi sur l'apprentissage ; l'AUC est mesurée sur la validation, jamais vue.
lignes = []
for c in VARIABLES:
    if train.loc[i_app, c].nunique() < 2:
        continue
    sens = 1 if roc_auc_score(y[i_app], train.loc[i_app, c]) >= 0.5 else -1
    lignes.append((c, sens, roc_auc_score(y[i_val], sens * train.loc[i_val, c])))
regles = (pd.DataFrame(lignes, columns=["variable", "sens", "AUC validation"])
          .sort_values("AUC validation", ascending=False).set_index("variable"))

# Hypothèse : la règle de la banque est celle dont l'AUC est la plus proche du plancher
REGLE = (regles["AUC validation"] - PLANCHER).abs().idxmin()
SENS = int(regles.loc[REGLE, "sens"])
y_val = y[i_val].to_numpy()
score_val = SENS * train.loc[i_val, REGLE].to_numpy()
print(f"validation : {len(i_val)} clients, {int(y_val.sum())} mécontents · règle métier : {REGLE} "
      f"(AUC {regles.loc[REGLE, 'AUC validation']:.4f})")

validation : 12164 clients, 481 mécontents · règle métier : var15 (AUC 0.6903)


In [8]:
regles.head(10)

,sens,AUC validation
variable,,
saldo_var30,-1,0.6936
var15,1,0.6903
saldo_var42,-1,0.6827
saldo_medio_var5_hace2,-1,0.6745
saldo_medio_var5_ult1,-1,0.6728
num_meses_var5_ult3,-1,0.6721
saldo_var5,-1,0.6718
saldo_medio_var5_ult3,-1,0.6686
num_var30,-1,0.6649


In [9]:
proches = regles[(regles["AUC validation"] - PLANCHER).abs() < 0.005]
print("Règles à moins de 0,005 du plancher :", list(proches.index))
print(f"Règle retenue : {REGLE}, sens {'+' if SENS > 0 else '-'} (plus {'âgé' if SENS > 0 else 'petit'} = plus de risque)")

Règles à moins de 0,005 du plancher : ['saldo_var30', 'var15']
Règle retenue : var15, sens + (plus âgé = plus de risque)


**Constat**

- `var15` (l'âge) prise seule donne **0,6903** sur notre validation, à 0,0013 du plancher de l'énoncé (0,6890). Plus le client est âgé, plus le risque est grand (jusqu'à 60 ans).
- `saldo_var30` fait un peu mieux (0,6936), mais l'écart est dans le bruit de mesure (± 0,01, voir P3).

**Décision**

- **Règle métier retenue (hypothèse) : trier les clients par âge, `var15`.** C'est la règle simple qui reproduit le plancher ; **à confirmer auprès de l'enseignant**.
- **Protocole : 80 / 20 stratifié, graine 42**, identique pour toute l'équipe. Tout essai se compare au plancher **sur la même validation**.